In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
sns.set_theme (style="whitegrid",palette="muted")
RANDOM_STATE = 42

In [ ]:
# Load the dataset
df_raw = pd.read_csv('/content/Z-Alizadeh sani dataset (2).csv')

# Assuming the last column is the target variable based on the problem description
X_raw = df_raw.iloc[:,:-1].copy()
y_raw = df_raw.iloc[:,-1].copy()

print("Feature matrix shape: ", X_raw.shape)
print("Target shape:", y_raw.shape)
display(X_raw.head())
display(y_raw.head())

Feature matrix shape:  (303, 54)
Target shape: (303,)


,Age,Weight,Length,Sex,BMI,DM,HTN,Current Smoker,EX-Smoker,FH,Obesity,CRF,CVA,Airway disease,Thyroid Disease,CHF,DLP,BP,PR,Edema,Weak Peripheral Pulse,Lung rales,Systolic Murmur,Diastolic Murmur,Typical Chest Pain,Dyspnea,Function Class,Atypical,Nonanginal,Exertional CP,LowTH Ang,Q Wave,St Elevation,St Depression,Tinversion,LVH,Poor R Progression,FBS,CR,TG,LDL,HDL,BUN,ESR,HB,K,Na,WBC,Lymph,Neut,PLT,EF-TTE,Region RWMA,VHD
0,53,90,175,Male,29.387755,0,1,1,0,0,Y,N,N,N,N,N,Y,110,80,0,N,N,N,N,0,N,0,N,N,N,N,0,0,1,1,N,N,90,0.7,250,155,30.0,8,7,15.6,4.7,141,5700,39,52,261,50,0,N
1,67,70,157,Fmale,28.398718,0,1,0,0,0,Y,N,N,N,N,N,N,140,80,1,N,N,N,N,1,N,0,N,N,N,N,0,0,1,1,N,N,80,1.0,309,121,36.0,30,26,13.9,4.7,156,7700,38,55,165,40,4,N
2,54,54,164,Male,20.077335,0,0,1,0,0,N,N,N,N,N,N,N,100,100,0,N,N,N,N,1,N,0,N,N,N,N,0,0,0,0,N,N,85,1.0,103,70,45.0,17,10,13.5,4.7,139,7400,38,60,230,40,2,mild
3,66,67,158,Fmale,26.838648,0,1,0,0,0,Y,N,N,N,N,N,N,100,80,0,N,N,N,Y,0,Y,3,N,Y,N,N,0,0,1,0,N,N,78,1.2,63,55,27.0,30,76,12.1,4.4,142,13000,18,72,742,55,0,Severe
4,50,87,153,Fmale,37.165193,0,1,0,0,0,Y,N,N,N,N,N,N,110,80,0,N,N,Y,N,0,Y,2,N,N,N,N,0,0,0,0,N,N,104,1.0,170,110,50.0,16,27,13.2,4.0,140,9200,55,39,274,50,0,Severe


,Cath
0,Cad
1,Cad
2,Cad
3,Normal
4,Normal


In [ ]:
# Preprocessing: Feature Analysis, Encoding, and Scaling
print("### 1. Full Feature-Level Analysis (from X_raw)")

feature_data = []
for col in X_raw.columns:
  unique_values_for_type_detection = X_raw[col].unique()
  num_unique = len(unique_values_for_type_detection)

  # Revised logic for detected_type to explicitly handle single-unique-value object columns
  if 2 <= num_unique <= 5:
    detected_type = 'Categorical'
  elif num_unique == 1:
    if X_raw[col].dtype == 'object':
      detected_type = 'Categorical' # Treat single unique object values as categorical
    else:
      detected_type = 'Numerical' # Treat single unique non-object values as numerical (constant)
  else: # num_unique > 5
    detected_type = 'Numerical'

  # Collect full unique values for display and missing values (this part should always run)
  unique_values_list = X_raw[col].unique().tolist()
  missing_values = X_raw[col].isnull().sum()

  # Limit unique values display to avoid clutter
  unique_val_display = str(unique_values_list) if len(unique_values_list) <= 10 else f'{len(unique_values_list)} unique values'

  feature_data.append({
      'Feature': col,
      'Pandas dtype': str(X_raw[col].dtype),
      '# Unique Values': num_unique,
      'Unique Values': unique_val_display,
      'Missing Values': missing_values,
      'Detected Type': detected_type
  })

feature_analysis_df = pd.DataFrame(feature_data)
display(feature_analysis_df)
print("\n### 2. Low-Cardinality Features (2-10 Unique Values)")
low_cardinality_features = feature_analysis_df[
    (feature_analysis_df['# Unique Values'] >= 2) &
    (feature_analysis_df['# Unique Values'] <= 10)
]
display(low_cardinality_features[['Feature','# Unique Values', 'Unique Values', 'Pandas dtype', 'Detected Type' ]])
print("\nFeatures detected as Categorical (2-5 unique values or 1 unique object value):",
      feature_analysis_df[feature_analysis_df['Detected Type'] == 'Categorical']['Feature'].tolist())
print("\nFeatures detected as Numerical (6+ unique values or 1 unique non-object value):",
      feature_analysis_df[feature_analysis_df['Detected Type'] == 'Numerical']['Feature'].tolist())


### 1. Full Feature-Level Analysis (from X_raw)


,Feature,Pandas dtype,# Unique Values,Unique Values,Missing Values,Detected Type
0,Age,int64,46,46 unique values,0,Numerical
1,Weight,int64,54,54 unique values,0,Numerical
2,Length,int64,44,44 unique values,0,Numerical
3,Sex,object,2,"['Male', 'Fmale']",0,Categorical
4,BMI,float64,262,262 unique values,0,Numerical
5,DM,int64,2,"[0, 1]",0,Categorical
6,HTN,int64,2,"[1, 0]",0,Categorical
7,Current Smoker,int64,2,"[1, 0]",0,Categorical
8,EX-Smoker,int64,2,"[0, 1]",0,Categorical
9,FH,int64,2,"[0, 1]",0,Categorical



### 2. Low-Cardinality Features (2-10 Unique Values)


,Feature,# Unique Values,Unique Values,Pandas dtype,Detected Type
3,Sex,2,"['Male', 'Fmale']",object,Categorical
5,DM,2,"[0, 1]",int64,Categorical
6,HTN,2,"[1, 0]",int64,Categorical
7,Current Smoker,2,"[1, 0]",int64,Categorical
8,EX-Smoker,2,"[0, 1]",int64,Categorical
9,FH,2,"[0, 1]",int64,Categorical
10,Obesity,2,"['Y', 'N']",object,Categorical
11,CRF,2,"['N', 'Y']",object,Categorical
12,CVA,2,"['N', 'Y']",object,Categorical
13,Airway disease,2,"['N', 'Y']",object,Categorical



Features detected as Categorical (2-5 unique values or 1 unique object value): ['Sex', 'DM', 'HTN', 'Current Smoker', 'EX-Smoker', 'FH', 'Obesity', 'CRF', 'CVA', 'Airway disease', 'Thyroid Disease', 'CHF', 'DLP', 'Edema', 'Weak Peripheral Pulse', 'Lung rales', 'Systolic Murmur', 'Diastolic Murmur', 'Typical Chest Pain', 'Dyspnea', 'Function Class', 'Atypical', 'Nonanginal', 'Exertional CP', 'LowTH Ang', 'Q Wave', 'St Elevation', 'St Depression', 'Tinversion', 'LVH', 'Poor R Progression', 'Region RWMA', 'VHD']

Features detected as Numerical (6+ unique values or 1 unique non-object value): ['Age', 'Weight', 'Length', 'BMI', 'BP', 'PR', 'FBS', 'CR', 'TG', 'LDL', 'HDL', 'BUN', 'ESR', 'HB', 'K', 'Na', 'WBC', 'Lymph', 'Neut', 'PLT', 'EF-TTE']


In [ ]:
print("### 3. Applying Label Encoding and Preparing Data for Scaling")
from sklearn.preprocessing import LabelEncoder

X_preprocessed = X_raw.copy()
label_encoders = {} # Store encoders and their mappings
feature_label_encoded = []

# Apply Label Encoding based on rules
for idx, row in feature_analysis_df.iterrows():
  col = row['Feature']
  if row['Detected Type'] == 'Categorical' and row ['Pandas dtype'] == 'object':
    le = LabelEncoder()
    X_preprocessed[col] = le.fit_transform(X_preprocessed[col])
    label_encoders[col] = {
        'encoder' : le,
        'mapping' : dict(zip(le.classes_, le.transform(le.classes_)))
    }
    feature_label_encoded.append(col)
    print(f" - Label Encoded feature '{col}' (object dtype, categorical). Mapping: {label_encoders[col]['mapping']}")

  # Identify columns for scaling (numerical features as detected by unique value count)
  numerical_col_for_scaling = feature_analysis_df[feature_analysis_df['Detected Type'] == 'Numerical']['Feature'].tolist()

  # Identify categorical columns that are NOT scaled (these are either non-object dtype or were label encoded)
# This includes features detected as categorical and non-object dtype which should be kept as is.
categorical_cols_not_scaled = [col for col in X_preprocessed.columns if col not in numerical_col_for_scaling]

print(f"\nFeatures identified for scaling (Numerical): {numerical_col_for_scaling}")
print(f"\nFeatures identified as categorical (either Label Encoded or kept as is): {categorical_cols_not_scaled}")

print("\nFirst 5 rows of X_processed after Label Encoding:")
display(X_preprocessed.head())

### 3. Applying Label Encoding and Preparing Data for Scaling
 - Label Encoded feature 'Sex' (object dtype, categorical). Mapping: {'Fmale': np.int64(0), 'Male': np.int64(1)}
 - Label Encoded feature 'Obesity' (object dtype, categorical). Mapping: {'N': np.int64(0), 'Y': np.int64(1)}
 - Label Encoded feature 'CRF' (object dtype, categorical). Mapping: {'N': np.int64(0), 'Y': np.int64(1)}
 - Label Encoded feature 'CVA' (object dtype, categorical). Mapping: {'N': np.int64(0), 'Y': np.int64(1)}
 - Label Encoded feature 'Airway disease' (object dtype, categorical). Mapping: {'N': np.int64(0), 'Y': np.int64(1)}
 - Label Encoded feature 'Thyroid Disease' (object dtype, categorical). Mapping: {'N': np.int64(0), 'Y': np.int64(1)}
 - Label Encoded feature 'CHF' (object dtype, categorical). Mapping: {'N': np.int64(0), 'Y': np.int64(1)}
 - Label Encoded feature 'DLP' (object dtype, categorical). Mapping: {'N': np.int64(0), 'Y': np.int64(1)}
 - Label Encoded feature 'Weak Peripheral Pulse' (object

,Age,Weight,Length,Sex,BMI,DM,HTN,Current Smoker,EX-Smoker,FH,Obesity,CRF,CVA,Airway disease,Thyroid Disease,CHF,DLP,BP,PR,Edema,Weak Peripheral Pulse,Lung rales,Systolic Murmur,Diastolic Murmur,Typical Chest Pain,Dyspnea,Function Class,Atypical,Nonanginal,Exertional CP,LowTH Ang,Q Wave,St Elevation,St Depression,Tinversion,LVH,Poor R Progression,FBS,CR,TG,LDL,HDL,BUN,ESR,HB,K,Na,WBC,Lymph,Neut,PLT,EF-TTE,Region RWMA,VHD
0,53,90,175,1,29.387755,0,1,1,0,0,1,0,0,0,0,0,1,110,80,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,90,0.7,250,155,30.0,8,7,15.6,4.7,141,5700,39,52,261,50,0,1
1,67,70,157,0,28.398718,0,1,0,0,0,1,0,0,0,0,0,0,140,80,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,80,1.0,309,121,36.0,30,26,13.9,4.7,156,7700,38,55,165,40,4,1
2,54,54,164,1,20.077335,0,0,1,0,0,0,0,0,0,0,0,0,100,100,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,85,1.0,103,70,45.0,17,10,13.5,4.7,139,7400,38,60,230,40,2,3
3,66,67,158,0,26.838648,0,1,0,0,0,1,0,0,0,0,0,0,100,80,0,0,0,0,1,0,1,3,0,1,0,0,0,0,1,0,0,0,78,1.2,63,55,27.0,30,76,12.1,4.4,142,13000,18,72,742,55,0,2
4,50,87,153,0,37.165193,0,1,0,0,0,1,0,0,0,0,0,0,110,80,0,0,0,1,0,0,1,2,0,0,0,0,0,0,0,0,0,0,104,1.0,170,110,50.0,16,27,13.2,4.0,140,9200,55,39,274,50,0,2


In [ ]:
# Encode the target
target_column =  y_raw.name # Corrected from y_raw.columns[0]
print("Target column:", target_column)
print("Unique labels:",y_raw.unique()) # Operate directly on y_raw

y =(y_raw.astype(str).str.strip().str.lower().map({"normal":0, "cad":1})) # Operate directly on y_raw

if y.isna().any():
  raise ValueError("Unexpected target labels. Inspect the unique values above.")

display(y.value_counts().rename(index={0:"normal",1:"CAD"})) # Corrected indentation

Target column: Cath
Unique labels: ['Cad' 'Normal']


,count
Cath,
CAD,216
normal,87


In [ ]:
# Train/Test Split
X_train, x_test, y_train, y_test = train_test_split(
 X_preprocessed, y,
   test_size=0.2, random_state= RANDOM_STATE,
   stratify=y
 )

print("Traning shape:",X_train.shape)
print("Testing shape:",x_test.shape)


Traning shape: (242, 54)
Testing shape: (61, 54)


In [ ]:
numerical_col_for_scaling

['Age',
 'Weight',
 'Length',
 'BMI',
 'BP',
 'PR',
 'FBS',
 'CR',
 'TG',
 'LDL',
 'HDL',
 'BUN',
 'ESR',
 'HB',
 'K',
 'Na',
 'WBC',
 'Lymph',
 'Neut',
 'PLT',
 'EF-TTE']

In [ ]:
scalar = StandardScaler()

# Separate numerical and categorical columns for training data
X_train_num = X_train[numerical_col_for_scaling]
X_train_cat = X_train[categorical_cols_not_scaled]

# Separate numerical and categorical columns for testing data
X_test_num = x_test[numerical_col_for_scaling]
X_test_cat = x_test[categorical_cols_not_scaled] # Corrected: should use categorical_cols_not_scaled

# Fit and transform the scaler only on numerical training data
X_train_scaled_num = scalar.fit_transform(X_train_num) # Corrected: fit_transform
# Transform the testing data using the scaler fitted on training data
X_test_scaled_num = scalar.transform(X_test_num) # Corrected: transform

# Convert scaled numerical arrays back to DataFrame for concatenation
X_train_scaled_num_df = pd.DataFrame(X_train_scaled_num, columns=numerical_col_for_scaling, index = X_train.index) # Corrected syntax and index
X_test_scaled_num_df = pd.DataFrame(X_test_scaled_num, columns=numerical_col_for_scaling, index = x_test.index) # Corrected syntax and index

# Reconstruct the full scaled dataframes by concatenating scaled numerical with unscaled categorical features
X_train_scaled = pd.concat([X_train_scaled_num_df,X_train_cat],axis=1)
X_test_scaled = pd.concat([X_test_scaled_num_df,X_test_cat],axis=1)

# Ensure column order is consistent (optional but good practice)
X_train_scaled = X_train_scaled[X_train.columns]
X_test_scaled = X_test_scaled[x_test.columns]

print("Scaled training shape:", X_train_scaled.shape)
print("First five feature means (from numerical scaled features):")
print(X_train_scaled_num[:, :5].mean(axis=0))
print("First five feature stds (from numerical scaled features):")
print(X_train_scaled_num[:, :5].std(axis=0))

print("\nFirst 5 rows of X_train_scaled (showing a mix of scaled numerical and original categorical values):")
display(X_train_scaled.head())

Scaled training shape: (242, 54)
First five feature means (from numerical scaled features):
[-1.90848255e-16  2.56911113e-17 -8.07434927e-17 -7.41372069e-16
  3.18386272e-16]
First five feature stds (from numerical scaled features):
[1. 1. 1. 1. 1.]

First 5 rows of X_train_scaled (showing a mix of scaled numerical and original categorical values):


,Age,Weight,Length,Sex,BMI,DM,HTN,Current Smoker,EX-Smoker,FH,Obesity,CRF,CVA,Airway disease,Thyroid Disease,CHF,DLP,BP,PR,Edema,Weak Peripheral Pulse,Lung rales,Systolic Murmur,Diastolic Murmur,Typical Chest Pain,Dyspnea,Function Class,Atypical,Nonanginal,Exertional CP,LowTH Ang,Q Wave,St Elevation,St Depression,Tinversion,LVH,Poor R Progression,FBS,CR,TG,LDL,HDL,BUN,ESR,HB,K,Na,WBC,Lymph,Neut,PLT,EF-TTE,Region RWMA,VHD
149,1.209246,-1.176117,-0.866040,0,-0.762928,0,1,0,0,1,0,0,0,0,0,0,1,0.033278,-1.673677,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,-0.729809,0.494819,-0.193833,0.533276,0.154939,2.295899,1.363500,0.252623,2.536226,-0.803619,3.200879,-2.637128,2.520652,0.586853,0.865624,0,1
25,0.626496,-0.043461,-0.097537,1,0.023815,0,0,0,0,0,1,0,0,0,0,0,0,-1.062418,0.564675,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,-0.576980,0.874364,-0.092037,0.060657,-0.697009,0.219970,-1.079308,1.023230,1.647353,0.539446,0.657611,-0.209365,0.969021,-0.489897,-0.240400,3,3
34,-0.344754,0.441963,0.231821,1,0.314478,0,0,0,0,0,1,0,0,0,0,1,1,-1.610266,2.803028,0,0,0,1,0,0,1,3,0,0,0,0,1,0,0,0,0,0,-0.596083,1.633452,0.925921,-0.467564,-0.507687,1.257934,-0.828764,0.445275,2.536226,-2.952524,0.949462,-0.209365,0.969021,-0.200621,-3.005462,4,0
176,-1.024628,1.008292,0.451394,1,0.755019,0,1,0,0,0,1,0,0,0,0,0,0,0.581126,-0.554501,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,-0.462357,0.115275,-0.010600,0.283066,0.912226,-0.817994,-0.452947,0.445275,0.980698,0.002220,-0.092862,0.318409,-0.996378,-0.136337,-0.240400,0,3
182,1.694871,0.846484,1.549255,1,-0.220490,0,1,0,0,0,1,0,0,0,0,0,0,0.033278,0.564675,0,0,0,0,0,0,1,2,0,0,0,0,0,0,0,0,0,0,-0.806224,-1.023358,-0.621375,-0.467564,-0.129044,-0.224872,-0.202403,0.637927,-1.241483,1.345286,3.451037,-2.214908,2.313768,-0.843456,0.312612,0,3
